# Exercise 4 — Triadic Knowledge Discovery

Triadic structure for this exercise:
- **Objects (G)**: the 200 sampled wines
- **Attributes (M)**: the 30 chemistry-only binary attributes from Exercise 2
  (the 3 quality-derived attributes are excluded from M since quality tier is
  now the *condition*, not an object property — including it would make B
  derivable from M and trivialize the triadic structure)
- **Conditions (B)**: quality tier `{Low, Medium, High}` (the same 3-tier
  split used in Exercise 1)
- **Incidence (Y)**: `(wine, property, tier) ∈ Y` iff the wine has that
  property **and** the wine's own quality tier equals `tier`

Note the consequence of this incidence definition up front: each wine
belongs to exactly one tier, so Y is a strict partition of objects across
conditions (no wine is incident under more than one tier). This shapes
every result below — it's flagged explicitly wherever it matters, rather
than glossed over.

In [1]:
import pandas as pd
import sys
sys.path.insert(0, ".")
from ex2_prepare_context import build_derived_context, all_attributes
import concepts
from itertools import combinations

sample = pd.read_csv("../data/wine_sample_200.csv")
derived = build_derived_context(sample)
attrs = all_attributes()
chem_attrs = [a for a in attrs if not a.startswith("quality")]
tiers = sample.set_index("wine_id")["quality_tier"].to_dict()
obj_attrs = {wid: set(a for a in derived[wid] if a in chem_attrs) for wid in derived}

print(f"{len(chem_attrs)} chemistry attributes (M)")
sample["quality_tier"].value_counts()


30 chemistry attributes (M)


quality_tier
Medium    165
High       27
Low         8
Name: count, dtype: int64

## 1. Triadic cross-table

Built as a long-format table: one row per `(wine, property)` pair that holds,
tagged with that wine's tier. This *is* the triadic incidence Y.

In [2]:
rows = []
for wid, aset in obj_attrs.items():
    tier = tiers[wid]
    for a in aset:
        rows.append((wid, a, tier))
triadic_table = pd.DataFrame(rows, columns=["wine_id", "property", "tier"])
print(f"{len(triadic_table)} (wine, property, tier) triples in Y")
triadic_table.head()


2732 (wine, property, tier) triples in Y


,wine_id,property,tier
0,1,sulphates>=low(0.4),High
1,1,chlorides=low_salt,High
2,1,fixed_acidity>=low(6),High
3,1,sulphates>=medium(0.6),High
4,1,sulphates>=high(0.8),High


## 2. Dyadic slices K_Low, K_Medium, K_High

Each slice is the formal context restricted to the wines in that tier (all
30 chemistry attributes kept). Concept lattices computed with `concepts`.

In [3]:
tier_wids = {t: [w for w, tt in tiers.items() if tt == t] for t in ["Low", "Medium", "High"]}
slices, lattices = {}, {}
for tier, wids in tier_wids.items():
    objects = [str(w) for w in wids]
    bools = [tuple(a in obj_attrs[w] for a in chem_attrs) for w in wids]
    ctx = concepts.Context(objects, chem_attrs, bools)
    slices[tier] = ctx
    lattices[tier] = ctx.lattice
    print(f"K_{tier}: {len(wids)} wines -> {len(ctx.lattice)} concepts")


K_Low: 8 wines -> 43 concepts


K_Medium: 165 wines -> 4781 concepts
K_High: 27 wines -> 599 concepts


## 3. Triconcepts

**Claim**: because Y is a strict object-tier partition (every wine belongs to
exactly one tier), every triconcept `(A, B, C)` with `A` nonempty has `|C| = 1`,
and the nontrivial triconcepts correspond exactly to the formal concepts of
the matching tier's dyadic slice, i.e. `(A, B, {tier})` is a triconcept iff
`(A, B)` is a formal concept of `K_tier`.

**Why**: if `A` is nonempty and `C` contains two distinct tiers `c1 != c2`,
then `A x B x C ⊆ Y` requires every `g ∈ A` to be incident under *both* `c1`
and `c2` — impossible, since each wine has only one tier. So any nonempty-`A`
triconcept has `|C| <= 1`. The `|C| = 0` case forces `A = G` trivially
uninteresting (no condition restricts it), and `|C| = 1` reduces the
incidence exactly to the dyadic slice for that one tier, where the usual
extent/intent maximality conditions apply unchanged. The only triconcept with
`A = ∅` is the (trivial) top: `(∅, M, {Low,Medium,High})`.

This is verified computationally below by checking a sample of candidate
triples directly against the maximality condition, rather than implementing
a general triadic NextClosure (which is well outside what's needed here once
the structural argument above is established).

In [4]:
def is_triconcept(A, B, C, obj_attrs, tiers):
    """Direct check of A x B x C subset Y and maximality against the
    actual Y, used only to spot-check the structural claim above."""
    for g in A:
        if {tiers[g]} != set(C):  # g's tier must equal EVERY condition in C
            return False
        if not (B <= obj_attrs[g]):
            return False
    return True

# spot-check: every formal concept of K_High, paired with C={"High"}, must be a triconcept
checked, ok = 0, 0
for c in lattices["High"]:
    A = {int(g) for g in c.extent}
    B = set(c.intent)
    checked += 1
    if is_triconcept(A, B, {"High"}, obj_attrs, tiers):
        ok += 1
print(f"{ok}/{checked} High-tier formal concepts verified as valid triconcepts (A,B,{{'High'}})")

# spot-check: the same (A,B) with C={"High","Medium"} must NOT satisfy A x B x C subset Y unless A is empty
violations = 0
for c in lattices["High"]:
    A = {int(g) for g in c.extent}
    B = set(c.intent)
    if A and is_triconcept(A, B, {"High", "Medium"}, obj_attrs, tiers):
        violations += 1
print(f"{violations} nonempty-extent High concepts remain valid when C is widened to two tiers (expected: 0)")


599/599 High-tier formal concepts verified as valid triconcepts (A,B,{'High'})
0 nonempty-extent High concepts remain valid when C is widened to two tiers (expected: 0)


## 4. Implication family 1 — Attribute implications `A1 ->_C A2`

"Under tier C, property A1 entails A2." Computed by running the Exercise-2
closure method independently on each tier's dyadic slice (sound and complete
for premise size <= 2, same method and same caveats as `ex2_implications.ipynb`).

In [5]:
def closure(context, props):
    return set(context.intension(context.extension(list(props))))


def small_premise_implications(context, max_premise=2):
    M = list(context.properties)
    results = []
    for size in range(1, max_premise + 1):
        for premise in combinations(M, size):
            pset = set(premise)
            clo = closure(context, pset)
            extra = clo - pset
            if not extra:
                continue
            if size == 2:
                a, b = premise
                if extra <= (closure(context, {a}) - {a}) | (closure(context, {b}) - {b}):
                    continue
            results.append((frozenset(pset), frozenset(extra)))
    return results


impls = {tier: small_premise_implications(slices[tier]) for tier in slices}
for tier in impls:
    print(tier, len(impls[tier]), "implications (premise size <= 2)")


Low 155 implications (premise size <= 2)
Medium 200 implications (premise size <= 2)
High 191 implications (premise size <= 2)


In [6]:
# Find an implication that holds under one tier (C) but not under another with the same premise
tier_specific = []
for p, q in impls["High"]:
    if len(p) > 2:
        continue
    if not slices["Medium"].extension(list(p)):
        continue
    med_closure = closure(slices["Medium"], p)
    if not (q <= med_closure):
        tier_specific.append((p, q, med_closure))

print(f"{len(tier_specific)} High-tier implications that do not hold (same premise) under Medium\n")
p, q, mc = tier_specific[0]
print(f"A1 ->_High A2:  {sorted(p)} -> {sorted(q)}")
print(f"Under Medium, the same premise only closes to: {sorted(mc)}")
print(f"\nSo 'fixed_acidity>=medium(8) ->_High citric_acid_present' holds, but")
print(f"'fixed_acidity>=medium(8) ->_Medium citric_acid_present' does NOT — a genuinely")
print(f"tier-conditioned implication, exactly the kind the A1 ->_C A2 family is meant to surface.")


163 High-tier implications that do not hold (same premise) under Medium

A1 ->_High A2:  ['fixed_acidity>=medium(8)'] -> ['alcohol>=9', 'citric_acid_present', 'fixed_acidity>=low(6)', 'sulphates>=low(0.4)']
Under Medium, the same premise only closes to: ['alcohol>=9', 'fixed_acidity>=low(6)', 'fixed_acidity>=medium(8)', 'sulphates>=low(0.4)']

So 'fixed_acidity>=medium(8) ->_High citric_acid_present' holds, but
'fixed_acidity>=medium(8) ->_Medium citric_acid_present' does NOT — a genuinely
tier-conditioned implication, exactly the kind the A1 ->_C A2 family is meant to surface.


## 5. Implication family 2 — Condition implications `b1 ->_A b2`

"Wines with properties A in tier b1 also have them in tier b2." Formally:
`forall g, forall m in A: (g,m,b1) in Y => (g,m,b2) in Y`.

Under our partition-style Y, `(g,m,b1) in Y` already requires `tiers[g] == b1`.
If `b1 != b2`, the consequent `(g,m,b2) in Y` requires `tiers[g] == b2` for the
*same* g — impossible unless no such g exists. So **this whole implication
family degenerates** under this incidence relation: `b1 ->_A b2` is
*vacuously true* whenever no wine in tier b1 satisfies all of A, and
*false* otherwise (for any b2 != b1). This is a structural consequence of
choosing "the wine's own tier" as the condition, not a bug — and it is worth
flagging as a modeling lesson: this incidence choice makes condition
implications uninformative as a discovery tool, even though attribute and
object implications (families 1 and 3) work fine.

In [7]:
def implication_b1_to_b2(A, b1, tier_wids, obj_attrs):
    satisfying = [w for w in tier_wids[b1] if A <= obj_attrs[w]]
    return len(satisfying) == 0, len(satisfying)  # (vacuously true?, count)


for label, A in [("alcohol>=13 (rare)", {"alcohol>=13"}), ("citric_acid_present (common)", {"citric_acid_present"})]:
    print(f"A = {label}")
    for b1 in ["Low", "Medium", "High"]:
        vacuous, count = implication_b1_to_b2(A, b1, tier_wids, obj_attrs)
        verdict = "VACUOUSLY TRUE for all b2" if vacuous else f"FALSE for any b2 != {b1} ({count} witnessing wines)"
        print(f"  {b1} ->_A b2 : {verdict}")
    print()


A = alcohol>=13 (rare)
  Low ->_A b2 : VACUOUSLY TRUE for all b2
  Medium ->_A b2 : FALSE for any b2 != Medium (2 witnessing wines)
  High ->_A b2 : FALSE for any b2 != High (2 witnessing wines)

A = citric_acid_present (common)
  Low ->_A b2 : FALSE for any b2 != Low (3 witnessing wines)
  Medium ->_A b2 : FALSE for any b2 != Medium (77 witnessing wines)
  High ->_A b2 : FALSE for any b2 != High (20 witnessing wines)



## 6. Implication family 3 — Object implications `g1 ->_C g2`

"Wine g1's properties are a subset of g2's, within tier C." Well-defined and
meaningful here (unlike family 2) since it only ever compares two wines
*within the same tier* — no cross-tier issue.

In [8]:
object_implications = {"Low": [], "Medium": [], "High": []}
for tier, wids in tier_wids.items():
    for g1 in wids:
        for g2 in wids:
            if g1 != g2 and obj_attrs[g1] < obj_attrs[g2]:
                object_implications[tier].append((g1, g2))

for tier in object_implications:
    print(f"{tier}: {len(object_implications[tier])} object-implication pairs (g1 ->_C g2)")

g1, g2 = object_implications["High"][0]
print(f"\nExample, High tier: wine {g1}'s properties (n={len(obj_attrs[g1])}) are a strict subset")
print(f"of wine {g2}'s (n={len(obj_attrs[g2])}); the only extra property wine {g2} has is:")
print(" ", sorted(obj_attrs[g2] - obj_attrs[g1]))


Low: 0 object-implication pairs (g1 ->_C g2)
Medium: 318 object-implication pairs (g1 ->_C g2)
High: 5 object-implication pairs (g1 ->_C g2)

Example, High tier: wine 6's properties (n=14) are a strict subset
of wine 12's (n=15); the only extra property wine 12 has is:
  ['alcohol>=11']


Note `Low` has **zero** object-implication pairs — with only 8 wines and 30
attributes, the Low-tier wines are chemically too diverse (no two share a
clean subset relationship). This itself is informative: it suggests the Low
tier is not chemically cohesive, unlike Medium and High.

## 7. Stable vs. tier-specific concepts

Comparing the three lattices' *intents* (the attribute side of each concept)
directly — a concept is "stable" if the same intent appears in all three
tiers' lattices, "tier-specific" if it only appears in one.

In [9]:
intents = {tier: set(frozenset(c.intent) for c in lattices[tier]) for tier in lattices}
common_all3 = intents["Low"] & intents["Medium"] & intents["High"]
only_high = intents["High"] - intents["Medium"] - intents["Low"]
only_medium = intents["Medium"] - intents["High"] - intents["Low"]
only_low = intents["Low"] - intents["Medium"] - intents["High"]

print(f"Intents common to all 3 tiers: {len(common_all3)}")
print(f"Intents unique to High:        {len(only_high)}")
print(f"Intents unique to Medium:      {len(only_medium)}")
print(f"Intents unique to Low:         {len(only_low)}")
print(f"\n(Low's lattice has only {len(intents['Low'])} concepts total, so its 'unique' count is bounded by that)")


Intents common to all 3 tiers: 4
Intents unique to High:        232
Intents unique to Medium:      4396
Intents unique to Low:         17

(Low's lattice has only 43 concepts total, so its 'unique' count is bounded by that)


In [10]:
print("A stable (tier-universal) intent, e.g.:")
print(" ", sorted(min(common_all3, key=len)))
print()
print("A High-only intent, e.g.:")
print(" ", sorted(min(only_high, key=len)))


A stable (tier-universal) intent, e.g.:
  ['alcohol>=9', 'fixed_acidity>=low(6)', 'free_so2>=low(6)', 'sulphates>=low(0.4)', 'volatile_acidity>=low(0.3)', 'volatile_acidity>=medium(0.5)']

A High-only intent, e.g.:
  ['alcohol>=11', 'alcohol>=9', 'free_so2>=low(6)', 'sulphates>=low(0.4)']
